In [ ]:
import torch
import torchvision.models as models
import torch.nn as nn

In [ ]:
# Recreate model
model = models.resnet50(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 14)

# Load weights
model.load_state_dict(torch.load("model_weights.pth"))

model.eval()

**LOAD IMAGE**

In [ ]:
from PIL import Image
import torchvision.transforms as transforms

# same transform used in training
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

image = Image.open("test.jpg").convert("RGB")
image = transform(image)

# add batch dimension
image = image.unsqueeze(0)

**RUN PREDICTION**

In [ ]:
with torch.no_grad():
    outputs = model(image)

# convert to probabilities
probs = torch.sigmoid(outputs)

**DISPLAY RESULTS**

In [ ]:
labels = [
    'Atelectasis','Cardiomegaly','Effusion','Infiltration',
    'Mass','Nodule','Pneumonia','Pneumothorax',
    'Consolidation','Edema','Emphysema','Fibrosis',
    'Pleural_Thickening','Hernia'
]

probs = probs.squeeze().numpy()

for i, label in enumerate(labels):
    if probs[i] > 0.5:
        print(f"{label}: {probs[i]:.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(Image.open("test.jpg"))
plt.axis("off")

predicted = [labels[i] for i in range(len(labels)) if probs[i] > 0.5]

plt.title("Predicted: " + ", ".join(predicted))
plt.show()